In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime, timezone, timedelta
import os
import struct

# ==========================================
# 1. 徹底的なデータ修復ロジック
# ==========================================
DB_PATH = "/data/cat.db"
SUMMARY_FILE = "/app/shared_summary/summary.txt"

def safe_float(v):
    """ bytes型を数値に、それ以外も数値に強制変換 """
    if v is None: return 0.0
    if isinstance(v, bytes):
        try:
            return struct.unpack('<d', v)[0] if len(v) == 8 else 0.0
        except: return 0.0
    try: return float(v)
    except: return 0.0

def get_db_conn():
    conn = sqlite3.connect(DB_PATH, timeout=30)
    conn.execute("PRAGMA journal_mode=WAL;")
    return conn

def load_all_data():
    with get_db_conn() as conn:
        df_raw = pd.read_sql("SELECT timestamp, weight FROM raw_data ORDER BY timestamp", conn)
        df_raw['timestamp'] = df_raw['timestamp'].apply(safe_float)
        df_raw['weight'] = df_raw['weight'].apply(safe_float)
        try:
            df_labels = pd.read_sql("SELECT start_ts, end_ts, label, cat_w, waste_w FROM labels ORDER BY start_ts DESC", conn)
            for col in ['start_ts', 'end_ts', 'cat_w', 'waste_w']:
                df_labels[col] = df_labels[col].apply(safe_float)
        except:
            df_labels = pd.DataFrame()

    if not df_raw.empty:
        unit = "s" if df_raw["timestamp"].max() < 1e11 else "ms"
        df_raw["datetime"] = pd.to_datetime(df_raw["timestamp"], unit=unit, utc=True).dt.tz_convert('Asia/Tokyo').dt.tz_localize(None)
        return df_raw, df_labels, unit
    return pd.DataFrame(), pd.DataFrame(), "s"

# ==========================================
# 2. 自動更新 & 書き出しロジック
# ==========================================
def calculate_event_weights(start_ts, end_ts):
    # 直前・直後のデータ取得件数を5件に変更
    before = df[df["timestamp"] < start_ts].tail(5)
    after = df[df["timestamp"] > end_ts].head(5)
    
    if before.empty or after.empty: 
        return 0.0, 0.0, 0.0
    
    base_in = before["weight"].mean()
    base_out = after["weight"].mean()
    
    event_data = df[(df["timestamp"] >= start_ts) & (df["timestamp"] <= end_ts)]
    if event_data.empty: 
        return 0.0, 0.0, 0.0
    
    # 戻り値の3番目：base_out - base_in で正の排泄量を算出するように修正
    return (
        round(event_data["weight"].max() - base_in, 1), # cat_w (猫の体重)
        round(base_in, 1),                             # base_in (比較用)
        round(base_out - base_in, 1)                    # waste_w (排泄量)
    )

def write_summary_txt():
    # 1. DBから最新のラベルデータを読み込む
    _, l_df, unit = load_all_data()
    if l_df.empty: return
    
    os.makedirs(os.path.dirname(SUMMARY_FILE), exist_ok=True)
    
    with open(SUMMARY_FILE, "w", encoding="utf-8") as f:
        for _, row in l_df.iterrows():
            # タイムスタンプ変換
            dt = pd.to_datetime(row['start_ts'], unit=unit, utc=True).tz_convert('Asia/Tokyo')
            
            # DB上のラベル（"auto" など）を取得
            base_label = str(row['label'])
            ww = safe_float(row['waste_w'])
            
            # --- 出力時に数値(waste_w)から判定キーワードを生成 ---
            if ww >= 40:
                display_label = f"{base_label}(うんち/poop)"
            elif ww >= 10:
                display_label = f"{base_label}(おしっこ/pee)"
            else:
                display_label = f"{base_label}(入室のみ)"

            # LINE検索がヒットするフォーマットで書き出し
            f.write(f"[{dt.strftime('%Y/%m/%d %H:%M:%S')}] 判定: [{display_label}] 猫体重: {row['cat_w']:.1f}g 排泄量: {ww:.1f}g\n")
            
    print(f"Summary updated with {len(l_df)} records.")

def purge_and_reauto():
    with get_db_conn() as conn:
        conn.execute("DROP TABLE IF EXISTS labels")
        conn.execute("CREATE TABLE labels (start_ts REAL, end_ts REAL, label TEXT, cat_w REAL, waste_w REAL)")
        conn.commit()
    auto_tagging_recent()
    update_ui()

def reauto():
    auto_tagging_recent()
    update_ui()
    
def auto_tagging_recent(hours=24):
    df = load_all_data()
    if df.empty:
        return 0

    # 直近N時間だけ抽出
    latest_ts = df["timestamp"].max()
    cutoff = latest_ts - (hours * 3600)

    df_recent = df[df["timestamp"] >= cutoff]

    # 既存ラベルの最大end_tsを取得
    with get_db_conn() as conn:
        row = conn.execute(
            "SELECT MAX(end_ts) FROM labels"
        ).fetchone()

    last_labeled_ts = row[0] if row and row[0] else 0

    df_recent = df_recent[df_recent["timestamp"] > last_labeled_ts]

    if df_recent.empty:
        print("No new data")
        return 0

    tags = []
    is_inside = False
    start_ts = None

    for i in range(len(df_recent)):
        w = df_recent.iloc[i]["weight"]
        ts = df_recent.iloc[i]["timestamp"]

        if not is_inside and w > 500:
            is_inside = True
            start_ts = ts
        elif is_inside and w < 500:
            tags.append((start_ts, ts))
            is_inside = False

    with get_db_conn() as conn:
        for s, e in tags:
            cw, ww = calculate_event_weights(df, s, e)

            if 3000 <= cw <= 4500:
                label = "うんち(poop)" if 40 <= ww <= 2000 else "おしっこ(pee)"
            elif cw > 500:
                label = f"入室のみ({cw:.0f}g)"
            else:
                continue

            conn.execute(
                "INSERT INTO labels VALUES (?,?,?,?,?)",
                (float(s), float(e), label, float(cw), float(ww))
            )

        conn.commit()

    print(f"Tagged {len(tags)} new events")
    return len(tags)
    
def auto_tagging_all():
    global df
    if df.empty: return 0
    tags, is_inside, start_ts = [], False, None
    for i in range(len(df)):
        w, ts = df.iloc[i]["weight"], df.iloc[i]["timestamp"]
        if not is_inside and w > 500: 
            is_inside, start_ts = True, ts
        elif is_inside and w < 500:
            tags.append((start_ts, ts))
            is_inside = False
            
    if tags:
        with get_db_conn() as conn:
            for s, e in tags:
                cw, _, ww = calculate_event_weights(s, e)
                
                # --- 体重がターゲット(4000g〜4500g)の時のみ詳細判定 ---
                if 3000 <= cw <= 4500:
                    # 40gから500gの間だけを「うんち」と定義
                    if 40 <= ww <= 2000:
                        final_label = "うんち(poop)"
                    else:
                        # マイナス値、40g未満、あるいは500g超の異常値も
                        # 実データに基づき「おしっこ」としてカウント
                        final_label = "おしっこ(pee)"
                
                # 体重が範囲外の場合
                elif cw > 500:
                    final_label = f"入室のみ({cw:.0f}g)"
                else:
                    continue # 500g以下のノイズは無視
                
                conn.execute("INSERT INTO labels VALUES (?,?,?,?,?)", 
                             (float(s), float(e), final_label, float(cw), float(ww)))
            conn.commit()
        write_summary_txt()
    return len(tags)


# ==========================================
# 3. UI 描画
# ==========================================
df, labels, ts_unit = load_all_data()
plot_out, list_out = widgets.Output(), widgets.Output(layout={'height': '200px', 'overflow_y': 'scroll'})
range_sld = widgets.SelectionRangeSlider(options=[('...', 0)], layout={'width': '95%'})
btn_refresh = widgets.Button(description="REFRESH", button_style='info')
btn_purge = widgets.Button(description="PURGE & RE-AUTO TAG", button_style='danger')

def draw_graph(_=None):
    with plot_out:
        clear_output(wait=True)
        if df.empty: return
        s, e = range_sld.value
        t_df = df[(df["timestamp"] >= s) & (df["timestamp"] <= e)]
        if t_df.empty: return
        fig, ax = plt.subplots(figsize=(12, 4))
        if not labels.empty:
            for _, r in labels.iterrows():
                if not (r['end_ts'] < s or r['start_ts'] > e):
                    # 【修正】Timestamp型なので .dt は使わない
                    ds = pd.to_datetime(r['start_ts'], unit=ts_unit, utc=True).tz_convert('Asia/Tokyo').tz_localize(None)
                    de = pd.to_datetime(r['end_ts'], unit=ts_unit, utc=True).tz_convert('Asia/Tokyo').tz_localize(None)
                    ax.axvspan(ds, de, color='orange', alpha=0.15)
        ax.plot(t_df["datetime"], t_df["weight"], 'b.-', lw=1)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
        plt.grid(True, alpha=0.2); plt.tight_layout(); plt.show()

def update_ui(_=None):
    global df, labels, ts_unit
    df, labels, ts_unit = load_all_data()
    
    if not df.empty:
        # --- 過去3日間に限定するロジックを追加 ---
        latest_ts = df["datetime"].max()
        three_days_ago = latest_ts - timedelta(days=3)
        
        # 過去3日分だけのデータを抽出してスライダーの選択肢にする
        mask = df["datetime"] > three_days_ago
        display_df = df[mask]
        
        if display_df.empty:
            display_df = df.tail(100) # 万が一空なら直近100件
            
        # スライダーの選択肢を生成 (表示ラベル, 実際の値)
        opts = [(dt.strftime('%m/%d %H:%M'), ts) for dt, ts in zip(display_df["datetime"], display_df["timestamp"])]
        
        range_sld.options = opts
        # 初期状態は最新の300件（またはデータ全量）を選択状態にする
        total_opts = len(opts)
        range_sld.index = (max(0, total_opts - 300), total_opts - 1)
    
    with list_out:
        clear_output()
        if not labels.empty:
            l_disp = labels.copy()
            # リスト表示も直近30件に絞る
            l_disp['time'] = pd.to_datetime(l_disp['start_ts'], unit=ts_unit, utc=True).dt.tz_convert('Asia/Tokyo').dt.strftime('%m/%d %H:%M')
            display(l_disp[['time', 'label', 'cat_w', 'waste_w']].head(30))
            
    draw_graph()

btn_refresh.on_click(update_ui)
btn_purge.on_click(lambda b: reauto())
range_sld.observe(draw_graph, names='value')

update_ui()
display(widgets.VBox([widgets.HBox([btn_refresh, btn_purge]), range_sld, plot_out, list_out]))
